In [7]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import recall_score, f1_score

from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.combine import SMOTETomek

from xgboost import XGBClassifier

from scipy.stats import randint, uniform


In [8]:
df = pd.read_csv("hospital_readmissions_30k.csv")

In [9]:
df.head()

,patient_id,age,gender,blood_pressure,cholesterol,bmi,diabetes,hypertension,medication_count,length_of_stay,discharge_destination,readmitted_30_days
0,1,74,Other,130/72,240,31.5,Yes,No,5,1,Nursing_Facility,Yes
1,2,46,Female,120/92,292,36.3,No,No,4,3,Nursing_Facility,No
2,3,89,Other,135/78,153,30.3,No,Yes,1,1,Home,No
3,4,84,Female,123/80,153,31.5,No,Yes,3,10,Home,No
4,5,32,Other,135/84,205,18.4,No,Yes,6,4,Nursing_Facility,No


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   patient_id             30000 non-null  int64  
 1   age                    30000 non-null  int64  
 2   gender                 30000 non-null  object 
 3   blood_pressure         30000 non-null  object 
 4   cholesterol            30000 non-null  int64  
 5   bmi                    30000 non-null  float64
 6   diabetes               30000 non-null  object 
 7   hypertension           30000 non-null  object 
 8   medication_count       30000 non-null  int64  
 9   length_of_stay         30000 non-null  int64  
 10  discharge_destination  30000 non-null  object 
 11  readmitted_30_days     30000 non-null  object 
dtypes: float64(1), int64(5), object(6)
memory usage: 2.7+ MB


In [11]:
df.isnull().sum()

,0
patient_id,0
age,0
gender,0
blood_pressure,0
cholesterol,0
bmi,0
diabetes,0
hypertension,0
medication_count,0
length_of_stay,0


In [12]:
df.duplicated().sum()

np.int64(0)

In [13]:
df[['systolic', 'diastolic']] = df['blood_pressure'].str.split('/', expand=True)

df['systolic'] = df['systolic'].astype(int)
df['diastolic'] = df['diastolic'].astype(int)


df = df.drop('blood_pressure', axis=1)

df.head()

,patient_id,age,gender,cholesterol,bmi,diabetes,hypertension,medication_count,length_of_stay,discharge_destination,readmitted_30_days,systolic,diastolic
0,1,74,Other,240,31.5,Yes,No,5,1,Nursing_Facility,Yes,130,72
1,2,46,Female,292,36.3,No,No,4,3,Nursing_Facility,No,120,92
2,3,89,Other,153,30.3,No,Yes,1,1,Home,No,135,78
3,4,84,Female,153,31.5,No,Yes,3,10,Home,No,123,80
4,5,32,Other,205,18.4,No,Yes,6,4,Nursing_Facility,No,135,84


In [14]:
le = LabelEncoder()
df['readmitted_30_days'] = le.fit_transform(df['readmitted_30_days'])
df['diabetes'] = le.fit_transform(df['diabetes'])
df['hypertension'] = le.fit_transform(df['hypertension'])

categorical_for_ohe = ['gender', 'discharge_destination']
df = pd.get_dummies(df, columns=categorical_for_ohe, drop_first=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 15 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   patient_id                              30000 non-null  int64  
 1   age                                     30000 non-null  int64  
 2   cholesterol                             30000 non-null  int64  
 3   bmi                                     30000 non-null  float64
 4   diabetes                                30000 non-null  int64  
 5   hypertension                            30000 non-null  int64  
 6   medication_count                        30000 non-null  int64  
 7   length_of_stay                          30000 non-null  int64  
 8   readmitted_30_days                      30000 non-null  int64  
 9   systolic                                30000 non-null  int64  
 10  diastolic                               30000 non-null  in

In [15]:
df = df.drop('patient_id', axis=1)

df.head()

,age,cholesterol,bmi,diabetes,hypertension,medication_count,length_of_stay,readmitted_30_days,systolic,diastolic,gender_Male,gender_Other,discharge_destination_Nursing_Facility,discharge_destination_Rehab
0,74,240,31.5,1,0,5,1,1,130,72,False,True,True,False
1,46,292,36.3,0,0,4,3,0,120,92,False,False,True,False
2,89,153,30.3,0,1,1,1,0,135,78,False,True,False,False
3,84,153,31.5,0,1,3,10,0,123,80,False,False,False,False
4,32,205,18.4,0,1,6,4,0,135,84,False,True,True,False


In [16]:
X = df.drop('readmitted_30_days', axis=1)
y = df['readmitted_30_days']

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (24000, 13)
Testing data: (6000, 13)


In [18]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# **Model Training**

In [19]:
#XGBoost
xgb_classifier = XGBClassifier(random_state=42,  eval_metric='logloss')
xgb_classifier.fit(X_train, y_train)

y_pred_xgb = xgb_classifier.predict(X_test)

accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
classification_rep_xgb = classification_report(y_test, y_pred_xgb)

print(f"\nModel Accuracy XGB: {accuracy_xgb:.4f}")
print("\nClassification Report XGB:\n", classification_rep_xgb)


Model Accuracy XGB: 0.8668

Classification Report XGB:
               precision    recall  f1-score   support

           0       0.87      0.99      0.93      5231
           1       0.11      0.01      0.01       769

    accuracy                           0.87      6000
   macro avg       0.49      0.50      0.47      6000
weighted avg       0.77      0.87      0.81      6000



In [20]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.combine import SMOTETomek
from xgboost import XGBClassifier
from scipy.stats import randint, uniform

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# **Grid Search Tuning**

In [21]:
models_tuning = {
    'XGBoost': XGBClassifier(random_state=42),
}

param_grids_tuning = {
    'XGBoost': {'learning_rate': [0.1], 'max_depth': [3, 5]},
}

for name, model in models_tuning.items():
    print(f"\nMemproses {name} (Tuning Saja)...")
    grid = GridSearchCV(model, param_grids_tuning[name], cv=3, scoring='f1', n_jobs=-1)
    grid.fit(X_train_scaled, y_train)

    y_pred = grid.predict(X_test_scaled)
    print(f"[{name}] Best Params: {grid.best_params_}")
    print(f"Model Accuracy {name}: {accuracy_score(y_test, y_pred):.4f}")
    print(f"Classification Report {name}:\n", classification_report(y_test, y_pred, zero_division=0))


Memproses XGBoost (Tuning Saja)...
[XGBoost] Best Params: {'learning_rate': 0.1, 'max_depth': 5}
Model Accuracy XGBoost: 0.8717
Classification Report XGBoost:
               precision    recall  f1-score   support

           0       0.87      1.00      0.93      5231
           1       0.00      0.00      0.00       769

    accuracy                           0.87      6000
   macro avg       0.44      0.50      0.47      6000
weighted avg       0.76      0.87      0.81      6000



# **SMOTE**

In [22]:
smt = SMOTETomek(random_state=42)
X_train_smt, y_train_smt = smt.fit_resample(X_train_scaled, y_train)

print(f"Jumlah data Kelas 1 sebelum SMOTETomek: {sum(y_train == 1)}")
print(f"Jumlah data Kelas 1 setelah SMOTETomek: {sum(y_train_smt == 1)}\n")

models_default = {
    'XGBoost': XGBClassifier(random_state=42),
}

for name, model in models_default.items():
    model.fit(X_train_smt, y_train_smt)
    y_pred = model.predict(X_test_scaled)

    print(f"\nModel Accuracy {name} (SMOTE Only): {accuracy_score(y_test, y_pred):.4f}")
    print(f"Classification Report {name} (SMOTE Only):\n", classification_report(y_test, y_pred, zero_division=0))

Jumlah data Kelas 1 sebelum SMOTETomek: 2905
Jumlah data Kelas 1 setelah SMOTETomek: 20973


Model Accuracy XGBoost (SMOTE Only): 0.8687
Classification Report XGBoost (SMOTE Only):
               precision    recall  f1-score   support

           0       0.87      1.00      0.93      5231
           1       0.19      0.01      0.01       769

    accuracy                           0.87      6000
   macro avg       0.53      0.50      0.47      6000
weighted avg       0.79      0.87      0.81      6000



# **Grid + Smote using Pipeline**

In [23]:
param_grids_pipeline = {
    'XGBoost': {'model__learning_rate': [0.1], 'model__max_depth': [3, 5]},
}

for name, model in models_default.items():
    print(f"\nMemproses {name} (Tuning + SMOTE)...")

    # Rangkai Pipeline: SMOTE lalu Model
    pipeline = ImbPipeline([
        ('smotetomek', SMOTETomek(random_state=42)),
        ('model', model)
    ])

    # Gunakan X_train_scaled (belum di-SMOTE), pipeline yang akan handle SMOTE di dalam fold
    grid_pipe = GridSearchCV(pipeline, param_grids_pipeline[name], cv=3, scoring='f1', n_jobs=-1)
    grid_pipe.fit(X_train_scaled, y_train)

    y_pred = grid_pipe.predict(X_test_scaled)

    print(f"[{name}] Best Params: {grid_pipe.best_params_}")
    print(f"Model Accuracy {name} (Tuning+SMOTE): {accuracy_score(y_test, y_pred):.4f}")
    print(f"Classification Report {name} (Tuning+SMOTE):\n", classification_report(y_test, y_pred, zero_division=0))


Memproses XGBoost (Tuning + SMOTE)...
[XGBoost] Best Params: {'model__learning_rate': 0.1, 'model__max_depth': 3}
Model Accuracy XGBoost (Tuning+SMOTE): 0.8698
Classification Report XGBoost (Tuning+SMOTE):
               precision    recall  f1-score   support

           0       0.87      1.00      0.93      5231
           1       0.12      0.00      0.01       769

    accuracy                           0.87      6000
   macro avg       0.50      0.50      0.47      6000
weighted avg       0.78      0.87      0.81      6000



# **Randomized Search Tuning**

In [24]:
param_dists_rs = {
    'XGBoost': {
        'learning_rate': uniform(0.01, 0.3),
        'max_depth': randint(3, 8)
    }
}

for name, model in models_tuning.items():
    print(f"\nMemproses {name} (Randomized Search Saja)...")
    random_search = RandomizedSearchCV(model, param_distributions=param_dists_rs[name],
                                       n_iter=5, cv=3, scoring='f1', n_jobs=-1, random_state=42)
    random_search.fit(X_train_scaled, y_train)

    y_pred = random_search.predict(X_test_scaled)
    print(f"[{name}] Best Params: {random_search.best_params_}")
    print(f"Model Accuracy {name} (RS Saja): {accuracy_score(y_test, y_pred):.4f}")
    print(f"Classification Report {name} (RS Saja):\n", classification_report(y_test, y_pred, zero_division=0))



Memproses XGBoost (Randomized Search Saja)...
[XGBoost] Best Params: {'learning_rate': np.float64(0.14777466758976016), 'max_depth': 7}
Model Accuracy XGBoost (RS Saja): 0.8712
Classification Report XGBoost (RS Saja):
               precision    recall  f1-score   support

           0       0.87      1.00      0.93      5231
           1       0.33      0.01      0.01       769

    accuracy                           0.87      6000
   macro avg       0.60      0.50      0.47      6000
weighted avg       0.80      0.87      0.81      6000



# **Randomized + SMOTE**

In [25]:
param_dists_pipeline_rs = {
    'XGBoost': {
        'model__learning_rate': uniform(0.01, 0.3),
        'model__max_depth': randint(3, 8)
    }
}

for name, model in models_default.items():
    print(f"\nMemproses {name} (Randomized Search + SMOTE)...")

    pipeline = ImbPipeline([
        ('smotetomek', SMOTETomek(random_state=42)),
        ('model', model)
    ])

    random_pipe = RandomizedSearchCV(pipeline, param_distributions=param_dists_pipeline_rs[name],
                                     n_iter=5, cv=3, scoring='f1', n_jobs=-1, random_state=42)
    random_pipe.fit(X_train_scaled, y_train)

    y_pred = random_pipe.predict(X_test_scaled)

    print(f"[{name}] Best Params: {random_pipe.best_params_}")
    print(f"Model Accuracy {name} (RS+SMOTE): {accuracy_score(y_test, y_pred):.4f}")
    print(f"Classification Report {name} (RS+SMOTE):\n", classification_report(y_test, y_pred, zero_division=0))


Memproses XGBoost (Randomized Search + SMOTE)...
[XGBoost] Best Params: {'model__learning_rate': np.float64(0.05679835610086079), 'model__max_depth': 5}
Model Accuracy XGBoost (RS+SMOTE): 0.8680
Classification Report XGBoost (RS+SMOTE):
               precision    recall  f1-score   support

           0       0.87      1.00      0.93      5231
           1       0.10      0.00      0.01       769

    accuracy                           0.87      6000
   macro avg       0.49      0.50      0.47      6000
weighted avg       0.77      0.87      0.81      6000



In [26]:
models = {
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss'),
}

# Parameter grid (Awalan model__ wajib karena menggunakan Pipeline)
param_dists = {

    'XGBoost': {
        'model__learning_rate': uniform(0.01, 0.2),
        'model__max_depth': randint(3, 7)
    },
}

# Eksekusi untuk setiap model
for name, model in models.items():
    print(f"\n---> Memproses {name}...")

    # Rangkai Pipeline: Undersampling jalan dulu untuk memotong Kelas 0, baru Model
    pipeline = ImbPipeline([
        ('undersampler', RandomUnderSampler(random_state=42)),
        ('model', model)
    ])

    # Konfigurasi Tuning dengan target utama metrik 'recall'
    random_search = RandomizedSearchCV(
        pipeline,
        param_distributions=param_dists[name],
        n_iter=5,  # Mencoba 5 kombinasi parameter acak
        cv=3,      # 3-Fold Cross Validation
        scoring='recall', # Fokus agar model pintar mengenali Kelas 1
        n_jobs=-1,
        random_state=42
    )

    # Proses Training
    # Data Kelas 0 akan dipotong secara dinamis di dalam pipeline ini
    random_search.fit(X_train_scaled, y_train)

    # Prediksi menggunakan data tes asli yang tidak di-undersample
    y_pred = random_search.predict(X_test_scaled)

    # Cetak hasil
    print(f"[{name}] Best Params: {random_search.best_params_}")
    print(f"Classification Report {name}:\n", classification_report(y_test, y_pred, zero_division=0))


---> Memproses XGBoost...
[XGBoost] Best Params: {'model__learning_rate': np.float64(0.09916655057071823), 'model__max_depth': 5}
Classification Report XGBoost:
               precision    recall  f1-score   support

           0       0.89      0.61      0.73      5231
           1       0.16      0.48      0.23       769

    accuracy                           0.60      6000
   macro avg       0.52      0.55      0.48      6000
weighted avg       0.80      0.60      0.66      6000



In [27]:
import numpy as np
from sklearn.metrics import recall_score, f1_score

y_pred_probs = xgb_classifier.predict_proba(X_test_scaled)[:, 1]

# 2. Siapkan daftar threshold yang ingin diuji (dari 15% sampai 50%)
thresholds = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

for t in thresholds:
    # Jika probabilitas >= threshold, jadikan 1. Jika tidak, jadikan 0.
    y_pred_custom = (y_pred_probs >= t).astype(int)

    # Hitung metrik spesifik untuk kelas 1
    recall_1 = recall_score(y_test, y_pred_custom, pos_label=1, zero_division=0)
    f1_1 = f1_score(y_test, y_pred_custom, pos_label=1, zero_division=0)
    acc = accuracy_score(y_test, y_pred_custom)

    print(f"Threshold: {t:.2f} | Accuracy: {acc:.2f} | Recall Kelas 1: {recall_1:.2f} | F1 Kelas 1: {f1_1:.2f}")

# Jika kamu ingin melihat laporan lengkap dari threshold terbaik (misal 0.25)
best_t = 0.25
y_best_custom = (y_pred_probs >= best_t).astype(int)
print(f"\nClassification Report XGBoost dengan Threshold {best_t}:\n")
print(classification_report(y_test, y_best_custom, zero_division=0))


Threshold: 0.15 | Accuracy: 0.71 | Recall Kelas 1: 0.27 | F1 Kelas 1: 0.19
Threshold: 0.20 | Accuracy: 0.78 | Recall Kelas 1: 0.15 | F1 Kelas 1: 0.15
Threshold: 0.25 | Accuracy: 0.82 | Recall Kelas 1: 0.09 | F1 Kelas 1: 0.11
Threshold: 0.30 | Accuracy: 0.84 | Recall Kelas 1: 0.05 | F1 Kelas 1: 0.07
Threshold: 0.35 | Accuracy: 0.85 | Recall Kelas 1: 0.03 | F1 Kelas 1: 0.05
Threshold: 0.40 | Accuracy: 0.86 | Recall Kelas 1: 0.01 | F1 Kelas 1: 0.03
Threshold: 0.45 | Accuracy: 0.86 | Recall Kelas 1: 0.01 | F1 Kelas 1: 0.02
Threshold: 0.50 | Accuracy: 0.87 | Recall Kelas 1: 0.01 | F1 Kelas 1: 0.01

Classification Report XGBoost dengan Threshold 0.25:

              precision    recall  f1-score   support

           0       0.87      0.92      0.90      5231
           1       0.14      0.09      0.11       769

    accuracy                           0.82      6000
   macro avg       0.51      0.50      0.50      6000
weighted avg       0.78      0.82      0.80      6000



In [36]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

# 1. Split Data Asli
X = df.drop('readmitted_30_days', axis=1)
y = df['readmitted_30_days']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================================================
# 2. PROSES HARD CUT (MANUAL UNDERSAMPLING) PADA DATA TRAIN
# =========================================================
# Gabungkan X_train dan y_train sementara ke dalam satu tabel
train_data = X_train.copy()
train_data['target'] = y_train

# Pisahkan antara pasien yang kembali (1) dan tidak kembali (0)
kelas_0 = train_data[train_data['target'] == 0]
kelas_1 = train_data[train_data['target'] == 1]

# Potong kelas 0 secara acak agar jumlahnya sama persis dengan kelas 1
kelas_0_dipotong = kelas_0.sample(n=len(kelas_1), random_state=42)

# Gabungkan kembali kelas 1 dan kelas 0 yang sudah dipotong, lalu acak urutannya
train_data_seimbang = pd.concat([kelas_0_dipotong, kelas_1]).sample(frac=1, random_state=42)

# Pisahkan kembali menjadi X_train dan y_train yang BARU dan SEIMBANG
X_train_balanced = train_data_seimbang.drop('target', axis=1)
y_train_balanced = train_data_seimbang['target']

# Cetak untuk membuktikan datanya sudah benar-benar seimbang
print("="*40)
print("JUMLAH DATA TRAINING SETELAH DI-CUT:")
print(y_train_balanced.value_counts())
print("="*40)

# =========================================================
# 3. SCALING & MODELING (Jauh lebih sederhana)
# =========================================================
# Lakukan scaling pada data yang sudah seimbang
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_balanced)
X_test_scaled = scaler.transform(X_test) # Test set tetap utuh

# Karena data sudah 50:50, kita TIDAK PERLU LAGI pakai class_weight='balanced' atau Pipeline
print("\nMelatih Model dengan Data Hasil Cut Manual...")

# Coba langsung pada XGBoost
xgb = XGBClassifier(random_state=42)
xgb.fit(X_train_scaled, y_train_balanced)
y_pred_xgb = xgb.predict(X_test_scaled)

print("\n--- HASIL XGBOOST ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}")
print(classification_report(y_test, y_pred_xgb, zero_division=0))

JUMLAH DATA TRAINING SETELAH DI-CUT:
target
0    2905
1    2905
Name: count, dtype: int64

Melatih Model dengan Data Hasil Cut Manual...

--- HASIL XGBOOST ---
Accuracy: 0.5342
              precision    recall  f1-score   support

           0       0.89      0.53      0.67      5231
           1       0.15      0.55      0.23       769

    accuracy                           0.53      6000
   macro avg       0.52      0.54      0.45      6000
weighted avg       0.79      0.53      0.61      6000

